In [ ]:
from rdkit import Chem
import numpy as np
import pandas as pd
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score
from rdkit import DataStructs
import matplotlib.pyplot as plt

In [ ]:
# PARQUET_PATH = ("../Dane/chembl_ml_dataset_04.parquet")
PARQUET_PATH = ("../Dane/chembl_ml_dataset_04_CHEMBL2147.parquet")
EPOCHS = 201

In [ ]:
morgan = GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = morgan.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [ ]:
df = pd.read_parquet(PARQUET_PATH)

df = df[df["pchembl_value"].notna()]
df = df[df["canonical_smiles"].notna()]
print("Targety: ", df["target_chembl_id"].unique())
print("Count: ", df["target_chembl_id"].value_counts())

In [ ]:
# df = df[df["target_chembl_id"] == "CHEMBL203"]

In [ ]:
class QSARDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fp = smiles_to_fp(row["canonical_smiles"])
        if fp is None:
            return self.__getitem__((idx + 1) % len(self.df))
        y = np.float32(row["pchembl_value"])
        return torch.tensor(fp), torch.tensor(y)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.MSELoss()
dataset = QSARDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

In [ ]:
fractions = [0.1, 0.3, 0.5, 0.7, 1.0]

train_scores = []
val_scores = []

for frac in fractions:
    size = int(len(train_ds) * frac)
    subset, _ = torch.utils.data.random_split(train_ds, [size, len(train_ds) - size])

    loader = DataLoader(subset, batch_size=256, shuffle=True)

    # nowy model
    model = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # krótki trening
    for epoch in range(10):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).view(-1, 1)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    # ewaluacja
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            out = model(X_batch).cpu().numpy().flatten()

            preds.extend(out)
            targets.extend(y_batch.numpy())

    r2 = r2_score(targets, preds)
    val_scores.append(r2)
    train_scores.append(None)  # opcjonalnie można policzyć

# wykres
plt.figure()
plt.plot(fractions, val_scores, marker='o')
plt.xlabel("Fraction of training data")
plt.ylabel("R2")
plt.title("Learning curve")
plt.show()

In [ ]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).view(-1, 1)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).view(-1, 1)
            out = model(X_batch)
            loss = criterion(out, y_batch)
            val_loss += loss.item()

    val_loss = val_loss / len(test_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

In [ ]:
plt.figure()
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.title("Loss curves")
plt.show()

In [ ]:

model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        out = model(X_batch).cpu().numpy().flatten()
        preds.extend(out)
        targets.extend(y_batch.numpy())

preds = np.array(preds)
targets = np.array(targets)

r2 = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))

# aktywny vs nieaktywny
threshold = 6.0
y_true_cls = (targets >= threshold).astype(int)
y_pred_cls = (preds >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_true_cls, y_pred_cls))
print("R2:", r2)
print("RMSE:", rmse)

In [ ]:
torch.save(model.state_dict(), "../Dane/mlp_01.pt")